# Session 02 Sandbox · The Q3 Expenditure Compliance Review

**ARI1101 Foundations of Data Science** · Data formats: CSV, JSON, XML and documents

## The question

Brightwater Logistics Ltd, a fictional company, has asked its data scientist a question:

> **How many payments above €5,000 were made in the last quarter (Q3 2026), and were they in line with the company's Expenditure Approval Policy?**

The answer is needed in a short report to the Board, with text, tables and a chart.

Answering it needs evidence from seven sources, in seven different formats: the policy (a Word document), the finance system's payment export (CSV), the bank's statement feed (JSON), the approval workflow export (XML), the approved vendor list (Excel), a set of emails (plain text) and the Board's own finance summary (PDF). The evidence has to be extracted, combined, checked and turned into a report.

## What you do and what the kit does

Doing all of that from scratch is not a first-session task. This notebook uses **ari1101kit**, a small library written for this unit, which hides the steps that are not today's topic. Today's topic is **data formats**, so that is the part you do yourself.

| You do (today's topic) | The kit does (hidden for now) |
|---|---|
| Inspect each source and identify its format and shape | Extract text, rules and tables from Word, Excel and PDF files |
| Read CSV, JSON and XML with `pandas`, `json` and `ElementTree` | Clean the data and convert currencies inside its own checks |
| Store extracted information in a suitable format (JSON or CSV) and read it back | Apply the policy rules to every transaction |
| Combine sources into one table and choose a format for it | Lay out the final report |
| Compare what each format keeps and loses | |

Later sessions use the same company and the same data. Each time, the kit hands a different step to you: cleaning and duplicates in the Scrub session, charts and statistics in Explore, and so on.

## Looking inside the black box

Nothing in the kit is secret. `kit.explain(function)` prints a plain-language description of any kit function, and `kit.source(function)` prints its Python code. Use them whenever you want to know how a step was done.

## Recording your answers

Each task ends with a call such as

```python
kit.answer("T3.1", value=..., reasoning="...")
```

which saves your answer and your reasoning in an **answer book**, a JSON file in the folder `ari1101_answers/`. Where possible the kit gives immediate feedback (✓, ✗ with a hint, or • when your lecturer will read the answer). The answers are **formative**: they are for your learning and for discussion, not for marks. At the end, `kit.submit()` lists anything you have missed and downloads the file so that you can upload it to the VLE.

Answering again with the same task code replaces your earlier answer, so you can improve an answer at any time.

---
## 0 · Setup

Run the next cell once per Colab session. It installs the kit from GitHub (about 20 seconds). If the last line of its output mentions an error, run it again; if it still fails, ask in the VLE forum.

In [ ]:
%pip install -q "git+https://github.com/charlieabela/ari1101kit@v0.2.0"

In [ ]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ari1101kit as kit

print("ari1101kit version:", kit.__version__)

Start your answer book. Replace the name with your own; it is used only to name your answer file.

In [ ]:
kit.start("S02", student="Your Name")

`kit.sources()` lists the evidence, and `kit.workspace()` copies the files into a folder called `brightwater_q3` next to this notebook. From now on you read the files from that folder with your own code, exactly as you would with files a client had sent you.

In [ ]:
display(kit.sources())
folder = kit.workspace()

---
## 1 · Survey the evidence

Before reading any file into Python, look at what it actually contains. `kit.peek(name)` prints the first lines of a file **exactly as stored**, without interpreting them. For binary documents (Word, Excel, PDF) it tells you so, because their bytes are not readable text.

Run the cell and look at each source in turn.

In [ ]:
for name in kit.sources()["source"]:
    print("=" * 30, name, "=" * 30)
    kit.peek(name, lines=8)
    print()

### Task T1.1 · Format and shape of each source

For each source, record its **format** (`csv`, `json`, `xml`, `docx`, `xlsx`, `txt` or `pdf`) and the **shape** of its data:

- `table`: rows and columns, every record with the same fields;
- `nested`: objects inside objects, or lists inside records;
- `tree`: elements inside elements, with tags and attributes;
- `document`: running text, possibly with tables and figures.

Complete the dictionary below. Two entries are filled in as examples.

In [ ]:
survey = {
    "policy":       {"format": "docx", "shape": "document"},
    "erp":          {"format": "csv",  "shape": "table"},
    "bank":         {"format": ___,    "shape": ___},
    "approvals":    {"format": ___,    "shape": ___},
    "vendors":      {"format": ___,    "shape": ___},
    "emails":       {"format": ___,    "shape": ___},
    "board_report": {"format": ___,    "shape": ___},
}
kit.answer("T1.1", value=survey)

### Task T1.2 · What can be read directly?

Answer in two or three sentences: which sources can you read with `pandas`, `json` or `ElementTree`, and which need extraction (here done by the kit)? Why can the review question not be answered from any single source?

In [ ]:
kit.answer("T1.2", reasoning="""
Write your answer here.
""")

---
## 2 · From a policy document to structured rules

The policy is a Word document written for people. To check transactions against it, its rules have to become **data**. The kit extracts them for you:

In [ ]:
rules = kit.extract.policy_rules()

print(type(rules), "of", len(rules), "items")
print(type(rules[0]))
print(json.dumps(rules[1], indent=2, ensure_ascii=False))

Each rule is a dictionary. Look at rule R2 above: `applies_to_eur` and `approvers` are **dictionaries inside the dictionary**, and `roles` is a **list** inside those. Not every rule has the same keys; print a few more (`rules[3]`, `rules[6]`) to see the differences. If you want to know how the kit found these values in the text, run `kit.explain(kit.extract.policy_rules)`.

### Task T2.1 · How many rules?

Record the number of rules and exceptions extracted. In the reasoning, state the Python type of `rules` and of one rule.

In [ ]:
kit.answer("T2.1", value=___, reasoning="rules is a ___ of ___")

### Task T2.2 · Store the rules as JSON and read them back

JSON can hold dictionaries, lists, numbers and text, so nested rules fit it naturally. Save the rules with `json.dump`, read the file back with `json.load`, and check with `==` whether the reloaded object is identical to the original.

*Hint:* `json.dump(obj, f, indent=2, ensure_ascii=False)` writes readable JSON and keeps the € sign as a character.

In [ ]:
with open("policy_rules.json", "w", encoding="utf-8") as f:
    json.dump(___, f, indent=2, ensure_ascii=False)

with open("policy_rules.json", encoding="utf-8") as f:
    rules_reloaded = ___

same = ___
print("Identical after the round trip:", same)
kit.answer("T2.2", value=same)

### Task T2.3 · Now try the same with CSV

Turn the rules into a DataFrame, save it as CSV, read it back, and look at the `approvers` value of rule R2 (row 1). Record that value and answer: what happened to the nested fields, and which format would you choose to store the rules?

In [ ]:
rules_df = pd.DataFrame(rules)
display(rules_df)

rules_df.to_csv("policy_rules.csv", index=False)
rules_csv = pd.read_csv("policy_rules.csv")

value = rules_csv.loc[1, "approvers"]
print(repr(value), type(value))

kit.answer("T2.3", value=value, reasoning="""
Write your answer here.
""")

### Task T2.4 · The threshold table

The policy also contains a table of approval bands. Tables are naturally rectangular, so this one *does* suit a DataFrame and CSV. The kit returns it exactly as written in the document, so every value is text.

Build a DataFrame from it, convert the `Approvers required` column to numbers with `pd.to_numeric`, check the dtypes, and save it as `thresholds.csv`.

In [ ]:
thresholds = pd.DataFrame(kit.extract.policy_thresholds())
print(thresholds.dtypes)

thresholds["Approvers required"] = ___
print(thresholds.dtypes)
display(thresholds)

thresholds.to_csv("thresholds.csv", index=False)
kit.answer("T2.4", value=thresholds)

---
## 3 · The ERP payments (CSV)

The finance system's export is a CSV file: one row per payment. Read it yourself with `pd.read_csv` from the `brightwater_q3` folder.

### Task T3.1 · Read and inspect

Read the file, display the first rows, print the dtypes, and record the shape.

In [ ]:
erp = pd.read_csv(___)
display(erp.head())
print(erp.dtypes)

kit.answer("T3.1", value=erp.shape)

The histogram below shows how the payment amounts are distributed, with the €5,000 threshold marked. It uses a logarithmic x-axis because the amounts range from a few euro to tens of thousands.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(erp["amount"], bins=[10 ** (x / 10) for x in range(15, 48)], color="#4a6fa5")
ax.set_xscale("log")
ax.axvline(5000, color="#c0504d", ls="--")
ax.text(5300, ax.get_ylim()[1] * 0.85, "€5,000", color="#c0504d")
ax.set_xlabel("amount (log scale)")
ax.set_ylabel("payments")
ax.set_title("ERP payments in Q3 2026")
plt.show()

### Task T3.2 · A first count

How many rows have an `amount` greater than 5000? Use a boolean filter and `len()` or `.sum()`. Do not worry about currency yet.

In [ ]:
count_any = ___
print(count_any)
kit.answer("T3.2", value=count_any)

### Task T3.3 · The count the policy asks for

Look at the `currency` column: some payments are in US dollars. Rule R7 says thresholds apply to the **euro equivalent**, at the rate stated in the rule. Find the rate in your `rules` list (not by typing the number), add an `amount_eur` column, and count the payments **above** €5,000.

*Hint:* `erp["amount"].where(erp["currency"] == "EUR", erp["amount"] * rate)` keeps euro amounts and converts the others.

In [ ]:
rate = ___   # take it from the R7 dictionary in `rules`
erp["amount_eur"] = ___

high = erp[___]
print(len(high), "payments above €5,000 in euro")
kit.answer("T3.3", value=len(high))

### Task T3.4 · The edge cases

Display the payment of exactly 5,000.00 and the USD payment of 5,300.00. Should either be part of the review? Justify each answer with the words of the policy.

In [ ]:
display(erp[(erp["amount"] == 5000) | ((erp["amount"] == 5300) & (erp["currency"] == "USD"))])

kit.answer("T3.4", reasoning="""
Write your answer here.
""")

---
## 4 · The bank feed (JSON)

The bank's API returned the statement as one JSON document. Load it with `json.load` and explore its structure before turning anything into a table.

### Task T4.1 · Top-level structure

In [ ]:
with open(folder / "bank_feed_q3.json", encoding="utf-8") as f:
    bank_doc = ___

print(type(bank_doc))
print(list(bank_doc.keys()))
print(bank_doc["statement_period"])
kit.answer("T4.1", value=list(bank_doc.keys()))

### Task T4.2 · The transactions list

The payments are in `bank_doc["transactions"]`. How many are there? Print the first one to see its nesting.

In [ ]:
transactions = ___
print(len(transactions))
print(json.dumps(transactions[0], indent=2))
kit.answer("T4.2", value=len(transactions))

### Task T4.3 · Flatten into a DataFrame

`amount` and `counterparty` are objects inside each transaction. Flatten the list with `pd.json_normalize` and record the column names.

In [ ]:
bank = ___
display(bank.head())
kit.answer("T4.3", value=list(bank.columns))

### Task T4.4 · Large payments in the bank feed

Convert to euro as in T3.3 (use `amount.value` and `amount.currency`) and count the transactions above €5,000. Your count will differ from T3.3. In the reasoning, suggest why before doing T4.5.

In [ ]:
bank["amount_eur"] = ___
bank_high = bank[___]
print(len(bank_high))

kit.answer("T4.4", value=len(bank_high), reasoning="""
Write your answer here.
""")

### Task T4.5 · Linking the bank to the ERP

The `remittance_info` text starts with the ERP reference (for example `TX-2026-01002`). Extract it into its own column with a regular expression, then find the references that appear more than once.

*Hint:* `bank["remittance_info"].str.extract(r"(TX-\d{4}-\d{5})", expand=False)` returns the reference, or `NaN` where there is none.

In [ ]:
bank["erp_ref"] = ___
dupes = bank[bank["erp_ref"].duplicated(keep=False)].sort_values("erp_ref")
display(dupes[["bank_ref", "value_date", "amount.value", "remittance_info"]])

duplicated_refs = sorted(dupes["erp_ref"].unique())
print("No ERP reference:", bank["erp_ref"].isna().sum(), "transaction(s)")
kit.answer("T4.5", value=duplicated_refs)

The duplicates and the payment without an ERP reference are exactly the kind of problem the Scrub session deals with. Today it is enough to have found them: the format (a remittance string inside a nested JSON record) is what made them findable.

---
## 5 · The approval workflow (XML)

The Procurement Workflow System exports one `<Request>` element per approval request. Each request has **attributes** (inside the opening tag) and **child elements**, including a list of `<Approver>` elements. Look at the raw text first with `kit.peek("approvals", lines=14)`.

### Task T5.1 · Parse and count

In [ ]:
root = ET.parse(folder / "approvals_pws_q3.xml").getroot()
print(root.tag, root.attrib)

requests = ___     # all <Request> children of the root
print(len(requests))
kit.answer("T5.1", value=len(requests))

### Task T5.2 · One row per request

Loop over the requests and build a list of dictionaries, then a DataFrame. For each request record:

- `erp_ref`, `amount` and `currency`, which are **attributes**: `req.get("erp_ref")`;
- `n_approvers`, the number of `<Approver>` elements: `len(req.findall("Approvers/Approver"))`;
- `roles`, the approvers' roles joined into one string, for example `"Budget Holder; Finance Manager"`.

Then convert `amount` to a number. Attributes are always text.

In [ ]:
rows = []
for req in requests:
    approvers = req.findall("Approvers/Approver")
    rows.append({
        "erp_ref": req.get("erp_ref"),
        "amount": ___,
        "currency": ___,
        "n_approvers": ___,
        "roles": "; ".join(a.get("role") for a in approvers),
    })

pws = pd.DataFrame(rows)
pws["amount"] = ___
print(pws.dtypes)
display(pws.head())
kit.answer("T5.2", value=pws)

### Task T5.3 · Requests with fewer than two approvers

In [ ]:
too_few = ___
display(too_few)
kit.answer("T5.3", value=len(too_few))

### Task T5.4 · Attributes and element text

Which values did you read from attributes, and which from element text? Which needed a type conversion, and why? (Look at one `<Approver>` element: where are the role, the decision, the date and the person's name?)

In [ ]:
first_approver = requests[0].find("Approvers/Approver")
print(first_approver.attrib, "| text:", first_approver.text)

kit.answer("T5.4", reasoning="""
Write your answer here.
""")

---
## 6 · Combine the evidence and choose a format

You now have three tables: `high` (ERP payments above €5,000), `bank` and `pws`. The review needs one table of high-value payments with their approval information.

### Task T6.1 · One table, one row per payment

Merge `high` with `pws` on `erp_ref`. Use `how="left"` so that payments **without** an approval request are kept (they are among the most important rows). Keep the columns that matter for the review.

In [ ]:
review = high.merge(pws[["erp_ref", "n_approvers", "roles"]], on=___, how=___)
review["n_approvers"] = review["n_approvers"].fillna(0).astype(int)

review = review[["erp_ref", "posting_date", "vendor_id", "vendor_name", "amount", "currency",
                 "amount_eur", "n_approvers", "roles"]]
display(review.head())
print(review.shape)
kit.answer("T6.1", value=review)

### Task T6.2 · Save it in two formats and compare

Save `review` as `q3_high_value.csv` and as `q3_high_value.json` (`orient="records"`, `indent=2`). Compare the file sizes, then read both back and compare their dtypes with the original.

In [ ]:
review.to_csv("q3_high_value.csv", index=False)
review.to_json("q3_high_value.json", orient="records", indent=2)

sizes = {"csv": Path("q3_high_value.csv").stat().st_size,
         "json": Path("q3_high_value.json").stat().st_size}
print(sizes)

back_csv = pd.read_csv("q3_high_value.csv")
back_json = pd.read_json("q3_high_value.json")
display(pd.DataFrame({"original": review.dtypes, "from CSV": back_csv.dtypes, "from JSON": back_json.dtypes}))

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.bar(sizes.keys(), sizes.values(), color=["#4a6fa5", "#e3a33b"])
ax.set_ylabel("bytes")
ax.set_title("The same table in two formats")
plt.show()

kit.answer("T6.2", value=sizes, reasoning="""
Write your answer here.
""")

### Task T6.3 · Extension: back to XML

Optional. Write `review` with `review.to_xml("q3_high_value.xml", index=False)` and open the file with `kit.peek`-style printing (`print(Path("q3_high_value.xml").read_text()[:800])`). How does its structure compare with the PWS export, where approvers were nested elements and key values were attributes?

In [ ]:
review.to_xml("q3_high_value.xml", index=False)
print(Path("q3_high_value.xml").read_text()[:800])

kit.answer("T6.3", reasoning="""
Write your answer here (optional task).
""")

---
## 7 · The black box: checking compliance

Applying eight rules and an exception to every payment is work for the Explore and Interpret sessions. Here the kit does it for you, using **the file you saved**. It reads the ERP references from your CSV and checks each payment against the policy, the approvals, the vendor list and the emails.

In [ ]:
findings = kit.pipeline.compliance_check("q3_high_value.csv")
display(findings[~findings["compliant"]][["erp_ref", "vendor_name", "amount_eur", "band", "issues"]])

If the kit reports that your table is missing transactions, go back to T3.3: the check can only examine what you give it.

### Task T7.1 · How many payments were not compliant?

In [ ]:
n_not_compliant = ___
kit.answer("T7.1", value=n_not_compliant)

### Task T7.2 · What the question misses

Rule R4 treats payments to the same supplier within 7 days as one expenditure. The kit can search **all** payments, including those below €5,000, for possible split purchases. It can also extract the emails.

In [ ]:
splits = kit.pipeline.split_purchases()
display(splits)

for m in kit.extract.emails():
    print(m["date"][:10], "|", m["from"], "|", m["subject"])
    print("   references:", m["erp_refs"], "| amounts:", m["amounts_eur"])

Read the email from the Marketing department in full with `print(kit.extract.emails()[0]["body"])`. Record the number of payments involved in possible split purchases. In the reasoning, explain what a review that only looks at payments **above** €5,000 would miss, and what the emails add that the structured data cannot.

In [ ]:
n_split_payments = ___
kit.answer("T7.2", value=n_split_payments, reasoning="""
Write your answer here.
""")

---
## 8 · The report

### Task T8.1 · Checking the Board report

The Board has already received a finance summary (PDF). The kit extracts its claims and its table:

In [ ]:
board = kit.extract.board_report()
print("Stated:", board["stated_high_value_count"], "payments of", board["stated_threshold"])
display(board["table"])

Compare the Board report's count with yours from T3.3, and explain the difference. Think about the threshold wording and about currency.

In [ ]:
kit.answer("T8.1", value={"stated": ___, "ours": ___}, reasoning="""
Write your answer here.
""")

### Task T8.2 · Your findings and the report

Write your findings for the Board in 3 to 6 sentences: the answer to the question, the main kinds of breach, and one limitation of the review. The kit then lays out a report with a summary, your text, a chart and the tables of non-compliant payments and split purchases, as HTML and as a Word document.

In [ ]:
my_findings = """
Write your findings here. Separate paragraphs with a blank line.
"""

files = kit.report.compliance_report(findings, splits, notes=my_findings, author="Your Name")

kit.answer("T8.2", value=my_findings, reasoning="""
Which format would you send to the Board, and why not CSV or JSON?
""")

---
## Submit

`kit.progress()` shows what you have answered. `kit.submit()` lists anything missing and, in Colab, downloads your answer book. Upload that JSON file to the VLE.

Colab deletes the files in this session when it disconnects, including your answer book, so submit (or download the `ari1101_answers` folder) before you close the notebook.

In [ ]:
display(kit.progress())
answer_file = kit.submit()

### Before the next session

Keep this notebook and your answer book. In the Scrub session you will take over the work that the kit did quietly today: the duplicates in the bank feed, the unrecorded payment, the vendor name variants and the currency conversion.